# Chapter 3: Hierarchical Clustering

This notebook accompanies **Chapter 3** of the lecture notes.

**Agenda**

☕ · 📊 · 🔗 · 🌳 · ✂️ · 🏁

**Next steps (take it from here):** 🌿 · 🎯

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import cdist
from scipy.cluster.vq import kmeans2
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import (
    check_single_linkage, check_complete_linkage,
    check_average_linkage, check_ward_distance,
    check_cophenetic_correlation,
)

## ☕ The Aroma Dataset

In Chapter 2 you ran K-Means on 1 797 coffee aroma fingerprints - ten brewing methods projected onto two aroma coordinates. K-Means required you to choose K upfront and gave you one flat partition. Now the question changes: **can we discover the relationships between brewing methods at every level of granularity, without committing to K in advance?**

Hierarchical clustering builds a tree of merges from the bottom up. You read off groupings by cutting the tree at whichever level tells the story you need - K = 2 through K = 1 797 from a single run.

> The scatter below shows all ten varieties. Some form tight islands, others overlap. Does a three-cluster cut make sense visually, and if so, which varieties would you group together?

<details><summary>Thought</summary>

At a coarse K = 3 level, the scatter suggests a hot-concentrated group (espresso, ristretto, lungo), a milk-based group (latte, flat white, cappuccino), and a slow-extraction group (cold brew, pour over, french press), with americano sitting between the first and third. The exact partition depends on the linkage criterion: single linkage will chain nearby points across group boundaries, while Ward and complete linkage will favor compact, balanced clusters. This is exactly why hierarchical clustering is useful here - you can inspect the tree and decide where the natural breaks are rather than guessing K.
</details>

In [ ]:
df     = pd.read_csv('coffee_aroma_2d.csv')
X      = df[['x1', 'x2']].values
labels = df['variety'].values
n      = len(X)
print(f'{n} samples, {len(np.unique(labels))} varieties')

fig, ax = plt.subplots(figsize=(10, 7))
for variety, color in _variety_colors.items():
    mask = labels == variety
    ax.scatter(X[mask, 0], X[mask, 1],
               c=color, s=8, linewidths=0, label=variety, alpha=0.75)

ax.set_xlabel('Aroma coordinate 1')
ax.set_ylabel('Aroma coordinate 2')
ax.set_title('1 797 coffee aroma fingerprints', fontsize=10)
ax.legend(ncol=2, frameon=False, fontsize=8, markerscale=2, labelcolor=_TEXT)
tufte_axis(ax)
plt.tight_layout()
plt.show()

**Observe:**
- Espresso and ristretto form a tight cluster in the lower-right of aroma space.
- Milk-based drinks (latte, flat white, cappuccino) overlap heavily in the upper-left.
- Cold brew, pour over, and french press spread along the left edge, partially overlapping with the milk group.
- Americano and lungo sit between the concentrated and slow-extraction groups.

## 📊 The Distance Matrix

Hierarchical clustering starts from the pairwise distance matrix. For 1 797 samples that matrix has 1 797 x 1 797 entries.

> How many bytes does a 1 797 x 1 797 float64 matrix occupy? Is that a concern at this scale? At what n would it become one?

<details><summary>Thought</summary>

Each float64 is 8 bytes, so 1 797 x 1 797 x 8 = 25 833 672 bytes, roughly 25.8 MB. Tiny for modern hardware. The O(n^2) scaling becomes painful around n = 50 000 (about 20 GB) and prohibitive at n = 100 000 (80 GB). For truly large datasets you would switch to approximate nearest-neighbor methods or sample-based approaches. At n = 1 797, computing and storing the full matrix is instant.
</details>

In [ ]:
D = cdist(X, X, 'euclidean')
print(f'Distance matrix shape: {D.shape}')
print(f'Memory: {D.nbytes:,} bytes ({D.nbytes / 1e6:.1f} MB)')

# Sort by variety so the block structure is visible
sort_idx = np.argsort(labels)
D_sorted = D[np.ix_(sort_idx, sort_idx)]
sorted_labels = labels[sort_idx]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(D_sorted, cmap='magma_r', aspect='equal')
ax.set_title('Pairwise Euclidean distances (sorted by variety)', fontsize=10)

# Variety boundary lines and labels
varieties_sorted = list(dict.fromkeys(sorted_labels))
boundaries = [0]
for v in varieties_sorted:
    boundaries.append(boundaries[-1] + np.sum(sorted_labels == v))

for b in boundaries[1:-1]:
    ax.axhline(b - 0.5, color=_BORDER, linewidth=0.5, alpha=0.7)
    ax.axvline(b - 0.5, color=_BORDER, linewidth=0.5, alpha=0.7)

ax.set_xticks([])
ax.set_yticks([(boundaries[i] + boundaries[i+1]) / 2
               for i in range(len(varieties_sorted))])
ax.set_yticklabels(varieties_sorted, fontsize=7)
ax.tick_params(axis='both', which='both', length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

**Observe:**
- Dark diagonal blocks confirm that within-variety distances are small.
- The espresso-ristretto block and the cold-brew-pour-over block show low cross-variety distances - these pairs are close neighbors in aroma space.
- The brightest off-diagonal regions are espresso vs. the milk-based varieties (latte, flat white, cappuccino) - the most distant pairs in the dataset.

## 🔗 Linkage Criteria

Agglomerative clustering merges the two clusters with the smallest inter-cluster distance at each step. The **linkage criterion** defines what "inter-cluster distance" means. The three exercises below implement the three classical choices.

Each function receives:
- `cluster_a` and `cluster_b` - integer index arrays (rows of `dist_matrix` belonging to each cluster)
- `dist_matrix` - the full n x n pairwise distance matrix

Useful operation: `dist_matrix[np.ix_(cluster_a, cluster_b)]` extracts the rectangular submatrix of all cross-cluster distances.

### 🔍 Single Linkage

> Single linkage defines the inter-cluster distance as the distance between the nearest pair of points, one from each cluster. If a handful of pour-over samples sit close to a few french-press samples in aroma space, single linkage will merge the two groups early - even if most pour-over and french-press samples are far apart. Why is this problematic for finding variety-level clusters?

<details><summary>Thought</summary>

Because several varieties overlap along the edges of their point clouds. A single pour-over sample with an unusual aroma profile might sit adjacent to a french-press sample, giving a very small minimum pairwise distance. Single linkage would merge the two groups based on this one coincidental proximity, ignoring the bulk of the distribution. The result is chaining: one group absorbs samples from the other one at a time as it grows, rather than merging two compact clusters at once. The dendrogram for single linkage will look asymmetric and ladder-like on this dataset.
</details>

Implement `single_linkage`: return the **minimum** entry in the cross-cluster submatrix.

Useful operations: `np.ix_(cluster_a, cluster_b)`, `np.min()`.

In [ ]:
# Test clusters: espresso vs cold brew - the two most dissimilar groups.
_ca = np.where(labels == 'espresso')[0]
_cb = np.where(labels == 'cold_brew')[0]


def single_linkage(cluster_a, cluster_b, dist_matrix):
    """Return the minimum pairwise distance between cluster_a and cluster_b."""
    # YOUR CODE HERE
    pass


check_single_linkage(single_linkage, _ca, _cb, D)

### 📌 Complete Linkage

> Complete linkage uses the farthest pair of points instead of the nearest. Two clusters only merge when their most distant pair falls within the current threshold. On the aroma data, how does this change which clusters form first compared to single linkage?

<details><summary>Thought</summary>

Complete linkage is resistant to merging until every sample in cluster A is close to every sample in cluster B. For the aroma data, the overlapping milk-based varieties will only merge once even their most distant samples are within range - a higher bar than single linkage's nearest-pair criterion. This tends to produce more compact, similarly-sized clusters. The downside is sensitivity to outliers: one cappuccino sample with an unusual aroma profile would inflate the maximum distance to the latte cluster and delay the merge. The dendrogram should show more balanced branching than single linkage.
</details>

Implement `complete_linkage`: return the **maximum** entry in the cross-cluster submatrix.

Useful operations: `np.ix_(cluster_a, cluster_b)`, `np.max()`.

In [ ]:
def complete_linkage(cluster_a, cluster_b, dist_matrix):
    """Return the maximum pairwise distance between cluster_a and cluster_b."""
    # YOUR CODE HERE
    pass


check_complete_linkage(complete_linkage, _ca, _cb, D)

### ⚖️ Average Linkage

> Average linkage averages all pairwise distances between the two clusters. It is less sensitive to the single nearest or farthest pair. For the aroma data, when would you expect average linkage to give a meaningfully different result than single or complete linkage?

<details><summary>Thought</summary>

Average linkage diverges most from single and complete when clusters have mixed density - one compact core with a few scattered members. On the aroma data, if the latte samples have tight within-group spacing but one latte drifts toward cappuccino territory, single linkage is fooled by that one sample while complete linkage is overcautious about it. Average linkage weights the outlier as just one of many pairs, diluting its influence. With 180 samples per variety, the effect is more pronounced than in a small dataset: extreme pairs are a tiny fraction of the total, so average linkage behaves very differently from single or complete.
</details>

Implement `average_linkage`: return the **mean** of all entries in the cross-cluster submatrix.

Useful operations: `np.ix_(cluster_a, cluster_b)`, `.mean()`.

In [ ]:
def average_linkage(cluster_a, cluster_b, dist_matrix):
    """Return the mean pairwise distance between cluster_a and cluster_b."""
    # YOUR CODE HERE
    pass


check_average_linkage(average_linkage, _ca, _cb, D)

## 🌳 The Dendrogram

A dendrogram is the tree diagram that records every merge in order. Each leaf is one sample; clusters merge bottom-up; the height of each horizontal bar shows the inter-cluster distance at that merge.

With 1 797 samples the full dendrogram has 1 797 leaves - unreadable as a plot. The cells below use scipy's `truncate_mode='lastp'` to show only the last 30 merges, collapsing earlier merges into leaf nodes labelled with their sample count. This is the standard way to visualise large dendrograms: the top of the tree is where the interesting structure lives.

Ward's method (`'ward'`) minimizes the increase in total within-cluster variance at each merge - you will implement it in the optional section.

In [ ]:
# Guard: only plot if all three linkage functions are implemented
if single_linkage(_ca, _cb, D) is None or complete_linkage(_ca, _cb, D) is None or average_linkage(_ca, _cb, D) is None:
    print('⬜ Implement single_linkage, complete_linkage, and average_linkage above first.')
else:
    _methods = ['single', 'complete', 'average', 'ward']
    _titles  = ['Single', 'Complete', 'Average', 'Ward']

    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
    for ax, method, title in zip(axes, _methods, _titles):
        Z = linkage(X, method=method)
        dendrogram(
            Z, ax=ax,
            truncate_mode='lastp', p=30,
            leaf_font_size=6,
            color_threshold=0,
            above_threshold_color=_ACCENT,
        )
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('(leaf count among last 30 merges)')
        ax.tick_params(axis='x', which='both', length=0, labelsize=6)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.spines['left'].set_visible(True)
        ax.spines['left'].set_color(_BORDER)
        ax.spines['left'].set_linewidth(0.8)
        ax.tick_params(axis='y', colors=_TERRA, length=3, width=0.8)

    plt.tight_layout()
    plt.show()


**Observe:**
- Single linkage shows chaining: one enormous cluster absorbs samples one at a time, producing a lopsided tree with tiny singleton leaves peeling off.
- Complete and average linkage produce more balanced trees; large vertical jumps indicate natural cut points.
- Ward linkage gives the cleanest structure, with a few well-separated subtrees merging symmetrically near the top.
- The biggest vertical gap in the Ward dendrogram often corresponds to the most meaningful number of clusters.

## ✂️ Cutting the Dendrogram

To extract flat cluster assignments, cut the tree at a given height or request exactly K clusters. Below we cut at K = 3 using single, complete, and Ward linkage, then compare with K-Means (K = 3) from Chapter 2.

> Which varieties switch clusters across linkage criteria? Which stay together regardless?

In [ ]:
# Guard: only plot if all three linkage functions are implemented
if single_linkage(_ca, _cb, D) is None or complete_linkage(_ca, _cb, D) is None or average_linkage(_ca, _cb, D) is None:
    print('⬜ Implement single_linkage, complete_linkage, and average_linkage above first.')
else:
    _Z_ward = linkage(X, method='ward')
    _Z_sing = linkage(X, method='single')
    _Z_comp = linkage(X, method='complete')

    # K-Means for comparison
    _km_centroids, _km_asgn = kmeans2(X, 3, minit='points', seed=42)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    _panels = [
        (_Z_sing, 'Single, K = 3'),
        (_Z_comp, 'Complete, K = 3'),
        (_Z_ward, 'Ward, K = 3'),
    ]
    cmap = plt.cm.get_cmap('tab10', 3)

    for ax, (Z_ref, title) in zip(axes[:3], _panels):
        asgn = fcluster(Z_ref, 3, criterion='maxclust')
        for ci in range(1, 4):
            mask = asgn == ci
            ax.scatter(X[mask, 0], X[mask, 1], color=cmap(ci - 1),
                       s=6, linewidths=0, alpha=0.6)
        ax.set_xlabel('Aroma coordinate 1')
        ax.set_ylabel('Aroma coordinate 2')
        ax.set_title(title, fontsize=10)
        tufte_axis(ax)

    # K-Means panel
    ax = axes[3]
    for ci in range(3):
        mask = _km_asgn == ci
        ax.scatter(X[mask, 0], X[mask, 1], color=cmap(ci),
                   s=6, linewidths=0, alpha=0.6)
    ax.set_xlabel('Aroma coordinate 1')
    ax.set_ylabel('Aroma coordinate 2')
    ax.set_title('K-Means, K = 3', fontsize=10)
    tufte_axis(ax)

    plt.tight_layout()
    plt.show()


**Observe:**
- Single linkage at K = 3 places nearly all samples in one cluster and peels off a few singletons or tiny groups - the chaining effect makes balanced three-way splits impossible.
- Complete and Ward linkage produce more balanced partitions. Espresso and ristretto consistently land in the same cluster; latte, flat white, and cappuccino stay together as well.
- Americano is the swing variety: it may join the concentrated group (Ward) or the slow-extraction group (complete) depending on the linkage criterion.
- The K-Means partition most closely resembles Ward, because both methods favor compact, spherical clusters. Where they disagree is at cluster boundaries where density is ambiguous.
- Hierarchical clustering gave us K = 1 through K = 1 797 from a single run; K-Means would need 1 797 separate runs.

### 🏁 Recap

**What we did:**
- ☕ Returned to the 1 797 aroma fingerprints from Chapter 2 and visualized the ten brewing methods.
- 📊 Computed the 1 797 x 1 797 distance matrix (25.8 MB) and confirmed block structure in the sorted heatmap.
- 🔗 Implemented single, complete, and average linkage - three definitions of inter-cluster distance.
- 🌳 Compared truncated dendrograms across all four linkage methods and saw chaining in single linkage vs. balanced trees in Ward.
- ✂️ Cut at K = 3, compared three hierarchical partitions with K-Means, and identified which varieties switch clusters across criteria.

**Key takeaways:**
- Hierarchical clustering encodes all possible K-partitions in one tree; you choose K after seeing the dendrogram.
- The linkage criterion shapes the result more than the distance metric in most practical cases.
- Ward linkage often agrees with K-Means on well-separated, compact clusters - both minimize within-cluster variance.
- Single linkage is useful for detecting outliers and elongated structures, but chains badly on globular data.

Next: **Chapter 4** introduces density-based clustering (DBSCAN), which handles clusters of arbitrary shape without requiring a distance threshold or cluster count.

---

## Take It from Here: Next Steps (Optional)

The exercises below are **optional** extensions. They are not required to follow Chapter 4. Work through them at your own pace after the session.

**Next steps:** 🌿 Ward distance · 🎯 Cophenetic correlation

### 🌿 Ward Distance: Minimizing Variance

> Ward linkage merges the pair of clusters whose union increases total within-cluster variance (WCSS) by the least. The espresso group has very low within-group spread (tight aroma profile). Does Ward merge it early or late, and why?

<details><summary>Thought</summary>

Late - and for a revealing reason. Merging the compact espresso cluster with any other group would cause a large WCSS increase because espresso is so far from all other varieties in aroma space. Ward's criterion postpones any merge that causes a big variance jump, so espresso sits as an isolated subtree until very near the root of the dendrogram. Within the milk-based group (latte, flat white, cappuccino), merges happen first because those varieties are close and the variance increase is small. The size-weighted formula also means that merging two large clusters is penalized even if their centroids are moderately close, giving Ward a natural preference for balanced trees.
</details>

Implement `ward_distance`: return the **increase in total WCSS** when merging two clusters.

Steps:
1. Compute WCSS of `cluster_a_pts` and `cluster_b_pts` separately.
2. Stack both and compute WCSS of the merged cluster.
3. Return `wcss_merged - wcss_a - wcss_b`.

WCSS of a set S: `np.sum((S - S.mean(axis=0)) ** 2)`.

Useful operations: `np.vstack()`, `.mean(axis=0)`, `np.sum()`.

In [ ]:
# Test: espresso vs cold brew (a merge Ward would postpone)
_pts_a = X[labels == 'espresso']
_pts_b = X[labels == 'cold_brew']


def ward_distance(cluster_a_pts, cluster_b_pts):
    """Return the increase in total WCSS when merging two clusters.

    Parameters
    ----------
    cluster_a_pts : ndarray, shape (n_a, d)
    cluster_b_pts : ndarray, shape (n_b, d)

    Returns
    -------
    float : wcss_merged - wcss_a - wcss_b
    """
    # YOUR CODE HERE
    pass


check_ward_distance(ward_distance, _pts_a, _pts_b)

### 🎯 Cophenetic Correlation: How Well Does the Dendrogram Preserve Distances?

> The cophenetic distance between two samples is the height at which they first join in the dendrogram. If the dendrogram faithfully reflects the original pairwise distances, cophenetic and original distances should correlate strongly. Which linkage method do you expect to score highest on the aroma data, and why?

<details><summary>Thought</summary>

Average linkage tends to score highest on cophenetic correlation across many datasets, because it uses the full distribution of pairwise distances between clusters rather than just the extreme values. Single linkage usually scores the lowest on compact, separated clusters like these: chaining makes faraway samples appear close in the tree (their cophenetic distance is small because a chain of short steps connects them), distorting the true large pairwise distances. Ward linkage sometimes scores lower than average even when it visually produces a more interpretable dendrogram - cophenetic correlation measures distance preservation, not cluster quality.
</details>

Implement `cophenetic_correlation`: return the Pearson correlation between the original pairwise distances and the cophenetic distances.

Steps:
1. For each pair (i, j), the cophenetic distance is the height of the first merge that places i and j in the same cluster.
2. Extract the upper triangle of both the original and cophenetic distance matrices (use `np.triu_indices(n, k=1)` to avoid double-counting).
3. Return `np.corrcoef(original_flat, cophenetic_flat)[0, 1]`.

Useful operations: `np.triu_indices(n, k=1)`, `np.corrcoef()`.

In [ ]:
def cophenetic_correlation(Z, dist_matrix):
    """Pearson correlation between original and cophenetic pairwise distances.

    Parameters
    ----------
    Z           : ndarray, shape (n-1, 4)  scipy linkage matrix
    dist_matrix : ndarray, shape (n, n)    original pairwise distances

    Returns
    -------
    float : cophenetic correlation coefficient in [-1, 1]
    """
    # YOUR CODE HERE
    pass


_Z_sing_opt = linkage(X, method='single')
_Z_avg_opt  = linkage(X, method='average')
_Z_ward_opt = linkage(X, method='ward')

for method_name, Z_ref in [('single', _Z_sing_opt), ('average', _Z_avg_opt), ('ward', _Z_ward_opt)]:
    print(f'  {method_name:10s}', end='  ')
    check_cophenetic_correlation(cophenetic_correlation, Z_ref, D)
